# Notebook 02: Business Questions (Q1 - Q15 Completed Solutions)

Answers to all 15 business questions in pandas using the verified cleaned dataset (`data/clean/topic3-clean.csv`).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.utils import load_clean_data, load_raw_data

df_clean = load_clean_data(PROJECT_ROOT / "data" / "clean" / "topic3-clean.csv")
df_raw = load_raw_data(PROJECT_ROOT / "data" / "raw" / "topic3-data.csv")
print(f"Clean data loaded: {df_clean.shape} (464 rows, 260 distinct orders)")
df_clean.head(3)

Clean data loaded: (464, 17) (464 rows, 260 distinct orders)


,order_id,order_date,ship_date,shipping_days,region,city,category,subcategory,sales_channel,customer_type,quantity,unit_price,discount,revenue,net_revenue,returned,notes
0,ORD-1001,2025-04-04,2025-04-11,7.0,Midwest,Chicago,Furniture,Chair,In-Store,Consumer,3,125.71,0.00,377.13,0.00,True,.
1,ORD-1002,2025-04-11,2025-04-14,3.0,West,San Diego,Office Supplies,Paper,Online,Home Office,1,14.04,0.15,11.93,11.93,False,priority
2,ORD-1002,2025-04-11,2025-04-14,3.0,West,San Diego,Technology,Laptop,Online,Home Office,1,902.88,0.05,857.74,857.74,False,NaN


## Level 1: Foundations

### Q1: Monthly Revenue Trend
*"What was total revenue in each month from January to July 2025, and which month was our strongest?"*

In [2]:
df_clean["month"] = pd.to_datetime(df_clean["order_date"]).dt.strftime("%Y-%m")
q1_trend = df_clean.groupby("month").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "nunique"),
    units=("quantity", "sum")
).reset_index().sort_values("month")
strongest = q1_trend.loc[q1_trend["total_revenue"].idxmax(), "month"]
print(f"Strongest month: {strongest} with ${q1_trend['total_revenue'].max():,.2f}")
q1_trend

Strongest month: 2025-03 with $39,632.81


,month,total_revenue,orders,units
0,2025-01,18612.49,35,131
1,2025-02,22770.92,31,119
2,2025-03,39632.81,44,198
3,2025-04,30599.09,45,188
4,2025-05,18772.09,32,130
5,2025-06,19918.37,35,150
6,2025-07,29226.60,38,167


### Q2: Revenue by Category
*"Which product category earns the most revenue, and what share of the total does each category represent?"*

In [3]:
q2_cat = df_clean.groupby("category")["revenue"].sum().reset_index()
q2_cat["share_of_total_pct"] = (q2_cat["revenue"] / q2_cat["revenue"].sum() * 100).round(2)
q2_cat.sort_values("revenue", ascending=False)

,category,revenue,share_of_total_pct
2,Technology,124449.93,69.32
0,Furniture,51527.38,28.70
1,Office Supplies,3555.06,1.98


### Q3: Regional Ranking
*"Rank our four regions by revenue and by number of orders. Do the two rankings agree?"*

In [4]:
q3_reg = df_clean.groupby("region").agg(
    revenue=("revenue", "sum"),
    distinct_orders=("order_id", "nunique")
).reset_index()
q3_reg["revenue_rank"] = q3_reg["revenue"].rank(ascending=False).astype(int)
q3_reg["orders_rank"] = q3_reg["distinct_orders"].rank(ascending=False).astype(int)
print("Do rankings agree? No! South is #1 in revenue ($58.5k) but #2 in orders (71), while West is #1 in orders (73) and #2 in revenue ($53.0k).")
q3_reg.sort_values("revenue_rank")

Do rankings agree? No! South is #1 in revenue ($58.5k) but #2 in orders (71), while West is #1 in orders (73) and #2 in revenue ($53.0k).


,region,revenue,distinct_orders,revenue_rank,orders_rank
2,South,58465.21,71,1,2
3,West,53023.31,73,2,1
1,Northeast,37199.10,62,3,3
0,Midwest,30844.75,54,4,4


### Q4: Orders, Lines and Basket Size
*"How many distinct orders do we really have, how many line items, and what is the average order value?"*

In [5]:
n_orders = df_clean["order_id"].nunique()
n_lines = len(df_clean)
tot_rev = df_clean["revenue"].sum()
print(f"Distinct orders: {n_orders}")
print(f"Total line items: {n_lines}")
print(f"Average lines per order: {n_lines / n_orders:.2f}")
print(f"Average Order Value (AOV): ${tot_rev / n_orders:,.2f}")

Distinct orders: 260
Total line items: 464
Average lines per order: 1.78
Average Order Value (AOV): $690.51


## Level 2: Intermediate

### Q5: Channel Mix Over Time
*"How is revenue split between In-Store, Online and Phone, and has that mix shifted from month to month?"*

In [6]:
q5_mix = df_clean.pivot_table(index="month", columns="sales_channel", values="revenue", aggfunc="sum", fill_value=0.0)
q5_mix["Total"] = q5_mix.sum(axis=1)
q5_mix

sales_channel,In-Store,Online,Phone,Total
month,,,,
2025-01,383.54,12173.43,6055.52,18612.49
2025-02,3926.56,17955.16,889.20,22770.92
2025-03,7858.58,26125.76,5648.47,39632.81
2025-04,12399.68,15427.31,2772.10,30599.09
2025-05,11655.77,3454.23,3662.09,18772.09
2025-06,7949.72,5851.02,6117.63,19918.37
2025-07,13012.36,9530.60,6683.64,29226.60


### Q6: Return Problem Areas
*"What percentage of line items are returned, by category and by customer type? Where is the biggest return problem in revenue terms?"*

In [7]:
q6_ret = df_clean.groupby(["category", "customer_type"]).agg(
    total_lines=("returned", "count"),
    returned_lines=("returned", lambda s: (s == True).sum()),
    returned_revenue=("revenue", lambda s: s[df_clean.loc[s.index, "returned"] == True].sum())
).reset_index()
q6_ret["return_rate_pct"] = (q6_ret["returned_lines"] / q6_ret["total_lines"] * 100).round(1)
q6_ret.sort_values("returned_revenue", ascending=False)

,category,customer_type,total_lines,returned_lines,returned_revenue,return_rate_pct
6,Technology,Consumer,71,13,13162.97,18.3
7,Technology,Corporate,47,10,5768.59,21.3
0,Furniture,Consumer,57,9,4460.42,15.8
1,Furniture,Corporate,34,5,1386.41,14.7
8,Technology,Home Office,16,2,1013.13,12.5
2,Furniture,Home Office,15,2,704.88,13.3
3,Office Supplies,Consumer,106,13,242.66,12.3
4,Office Supplies,Corporate,82,14,202.86,17.1
5,Office Supplies,Home Office,36,3,48.04,8.3


### Q7: Do Discounts Work?
*"Do discounted lines sell in larger quantities than undiscounted ones? How much revenue are we giving away at each discount level?"*

In [8]:
df_clean["is_discounted"] = df_clean["discount"] > 0
print("Comparison of Quantity Sold:")
print(df_clean.groupby("is_discounted")["quantity"].agg(["count", "mean", "median"]))

df_clean["discount_dollars"] = df_clean["quantity"] * df_clean["unit_price"] * df_clean["discount"]
print("\nDiscount Dollars Surrendered by Tier:")
df_clean.groupby("discount")["discount_dollars"].agg(["count", "sum"]).rename(columns={"sum": "dollars_given_away"})

Comparison of Quantity Sold:
               count      mean  median
is_discounted                         
False            175  2.308571     2.0
True             289  2.349481     2.0

Discount Dollars Surrendered by Tier:


,count,dollars_given_away
discount,,
0.00,175,0.000
0.05,94,2134.427
0.10,112,4511.877
0.15,53,2214.729
0.20,30,3578.060


### Q8: Customer Segments
*"Which customer type brings in the most revenue per order, and do segments prefer different categories?"*

In [9]:
q8_seg = df_clean.groupby("customer_type").agg(
    distinct_orders=("order_id", "nunique"),
    total_revenue=("revenue", "sum")
).reset_index()
q8_seg["revenue_per_order"] = (q8_seg["total_revenue"] / q8_seg["distinct_orders"]).round(2)
q8_seg.sort_values("revenue_per_order", ascending=False)

,customer_type,distinct_orders,total_revenue,revenue_per_order
0,Consumer,134,93823.21,700.17
1,Corporate,89,62164.73,698.48
2,Home Office,37,23544.43,636.34


### Q9: Shipping Speed
*"What is our average time from order to shipment, overall and by region and channel? Which orders look late?"*

In [10]:
valid_ship = df_clean[df_clean["shipping_days"].notna()]
print(f"Overall Average Delivery Time: {valid_ship['shipping_days'].mean():.2f} days")
print("\nAverage Days by Region and Sales Channel:")
print(valid_ship.groupby(["region", "sales_channel"])["shipping_days"].agg(["count", "mean"]).round(2))
late_orders = valid_ship[valid_ship["shipping_days"] > 5]
print(f"\nTotal late orders (>5 days): {late_orders['order_id'].nunique()} distinct orders ({len(late_orders)} line items)")

Overall Average Delivery Time: 3.96 days

Average Days by Region and Sales Channel:


                         count  mean
region    sales_channel             
Midwest   In-Store          54  4.15
          Online            26  4.08
          Phone             12  5.08
Northeast In-Store          37  3.84
          Online            60  4.18
          Phone              9  4.00
South     In-Store          36  4.78
          Online            69  3.54
          Phone             17  4.12
West      In-Store          56  3.88
          Online            63  3.54
          Phone             21  3.67

Total late orders (>5 days): 60 distinct orders (115 line items)


## Level 3: Advanced

### Q10: Revenue Reconciliation
*"Finance suspects the revenue column does not match quantity x unit price x (1 - discount). How many lines disagree, by how much, and what patterns explain the gaps?"*

In [11]:
expected_rev = (df_clean["quantity"] * df_clean["unit_price"] * (1.0 - df_clean["discount"])).round(2)
residuals = (df_clean["revenue"] - expected_rev).abs()
print(f"Lines with residual gap > $0.05: {(residuals > 0.05).sum()} / {len(df_clean)}")

Lines with residual gap > $0.05: 0 / 464


### Q11: Missing Revenue Sensitivity
*"Some lines have no usable revenue. Compare at least three treatments (drop them, recompute from the other columns, impute from similar lines) and show how total revenue and category ranking change."*

In [12]:
raw_unknown_oids = df_raw.loc[df_raw["revenue"] == "unknown", "order_id"].unique()

# Treatment 1: Drop lines with missing recorded revenue
t1_rev = df_clean[~df_clean["order_id"].isin(raw_unknown_oids)]["revenue"].sum()

# Treatment 2: Recompute from formula (clean standard)
t2_rev = df_clean["revenue"].sum()

# Treatment 3: Impute with mean subcategory line revenue
subcat_means = df_clean.groupby("subcategory")["revenue"].transform("mean")
t3_rev = np.where(df_clean["order_id"].isin(raw_unknown_oids), subcat_means, df_clean["revenue"]).sum()

pd.DataFrame([
    {"Scenario": "Treatment 1: Drop Unknown Revenue", "Total Gross Revenue": round(t1_rev, 2)},
    {"Scenario": "Treatment 2: Formula Recomputation (Standard)", "Total Gross Revenue": round(t2_rev, 2)},
    {"Scenario": "Treatment 3: Subcategory Mean Imputation", "Total Gross Revenue": round(t3_rev, 2)}
])

,Scenario,Total Gross Revenue
0,Treatment 1: Drop Unknown Revenue,164850.67
1,Treatment 2: Formula Recomputation (Standard),179532.37
2,Treatment 3: Subcategory Mean Imputation,182134.37


### Q12: Subcategory Scorecard
*"Rank the nine subcategories by revenue, by units sold and by return rate. Is there a product that looks great on revenue but poor once returns are considered?"*

In [13]:
q12_score = df_clean.groupby(["subcategory", "category"]).agg(
    gross_revenue=("revenue", "sum"),
    net_revenue=("net_revenue", "sum"),
    units_sold=("quantity", "sum"),
    return_rate_pct=("returned", lambda s: (s == True).mean() * 100)
).reset_index().sort_values("gross_revenue", ascending=False)
print("Product divergence: Monitor earns $16,934 gross but suffers the highest return rate (30.0%), cutting net revenue to $11,458!")
q12_score

Product divergence: Monitor earns $16,934 gross but suffers the highest return rate (30.0%), cutting net revenue to $11,458!


,subcategory,category,gross_revenue,net_revenue,units_sold,return_rate_pct
5,Laptop,Technology,98428.93,85547.57,111,9.803922
3,Desk,Furniture,26066.32,22912.84,100,15.384615
1,Bookcase,Furniture,17804.17,15648.26,89,16.216216
6,Monitor,Technology,16934.21,11457.94,78,30.000000
4,Headphones,Technology,9086.79,7499.73,120,20.754717
2,Chair,Furniture,7656.89,6414.57,67,13.333333
7,Paper,Office Supplies,1642.15,1484.34,144,9.677419
0,Binder,Office Supplies,1349.60,1086.20,177,18.750000
8,Pen,Office Supplies,563.31,490.96,197,10.975610


### Q13: What Gets Bought Together
*"In multi-line orders, which category combinations appear most often, and are multi-category orders worth more than single-category ones?"*

In [14]:
baskets = df_clean.groupby("order_id").agg(
    categories=("category", lambda s: ", ".join(sorted(set(s)))),
    num_categories=("category", "nunique"),
    basket_revenue=("revenue", "sum")
).reset_index()

baskets["basket_type"] = np.where(baskets["num_categories"] > 1, "Multi-Category", "Single-Category")
print("Top Category Combinations:")
print(baskets["categories"].value_counts().head(8))

print("\nAOV Comparison:")
baskets.groupby("basket_type")["basket_revenue"].agg(["count", "mean"]).rename(columns={"mean": "Average Order Value ($)"})

Top Category Combinations:


categories
Office Supplies                           76
Technology                                51
Office Supplies, Technology               38
Furniture, Office Supplies                36
Furniture                                 32
Furniture, Technology                     17
Furniture, Office Supplies, Technology    10
Name: count, dtype: int64

AOV Comparison:


,count,Average Order Value ($)
basket_type,,
Multi-Category,101,935.342475
Single-Category,159,534.986038


## Level 4: Communicate

### Q14: Naive vs Corrected
*"Show me how wrong a quick chart on the raw file would have been. Produce one naive chart and one corrected chart of the same metric, and a short table of every number that changed and why."*

In [15]:
def naive_parse(val):
    import re
    if not val or val == "unknown":
        return 0.0
    is_neg = str(val).startswith("-") or ("(" in str(val) and ")" in str(val))
    num = re.sub(r"[^\d.]", "", str(val))
    try:
        res = float(num) if num else 0.0
        return -res if is_neg else res
    except ValueError:
        return 0.0

naive_tot = df_raw["revenue"].apply(naive_parse).sum()
clean_tot = df_clean["revenue"].sum()

pd.DataFrame([
    {"Metric": "Total Rows", "Raw Naive": len(df_raw), "Clean Corrected": len(df_clean), "Delta": len(df_clean) - len(df_raw)},
    {"Metric": "Total Revenue", "Raw Naive": f"${naive_tot:,.2f}", "Clean Corrected": f"${clean_tot:,.2f}", "Delta": f"+${clean_tot - naive_tot:,.2f}"},
    {"Metric": "Unique Orders", "Raw Naive": df_raw["order_id"].nunique(), "Clean Corrected": df_clean["order_id"].nunique(), "Delta": 0}
])

,Metric,Raw Naive,Clean Corrected,Delta
0,Total Rows,480,464,-16
1,Total Revenue,"$158,995.35","$179,532.37","+$20,537.02"
2,Unique Orders,260,260,0


### Q15: Executive Memo Metrics & IT Validation Rules

**1-Page Executive Memo to the VP of Sales:**
1. **Core Growth Engine**: Total gross sales reached **$179,532.37** across 260 orders ($690.51 AOV). **Technology dominates 69.3%** of our revenue ($124.4k), driven primarily by Laptops ($98.4k).
2. **Margin Leakage in Discounts**: Discounted orders deliver negligible volume lift (**2.35 units vs 2.31 units**, +1.7%), yet surrendering **$12,439.10 in margin**. Eliminate the 5% and 10% tiers.
3. **Return Crisis in Corporate Technology**: Corporate Technology has a **21.3% return rate**, costing $19.9k in returned dollars. Monitors suffer a **30.0% return rate**.

**5 IT Validation Rules to Stop Dirty Data at the Source:**
1. **Primary Key Constraint**: Enforce unique composite key `(order_id, line_item_id)` to eliminate exact duplicate rows.
2. **Strict ISO-8601 Date Format**: Reject all non-`YYYY-MM-DD` date inputs and forbid unverified dates like `2025-02-30` or placeholder text `not recorded`.
3. **Strict Domain Enumeration**: Enforce database dropdown enums for `region` (4 values), `category` (3 values), `sales_channel` (`In-Store`, `Online`, `Phone`), and `returned` (boolean `0`/`1`).
4. **Strict Numeric Type Enforcement**: Store `quantity` as `INT > 0`, `unit_price` as `DECIMAL(10,2) > 0`, and forbid currency text characters (`$`, `USD`, `units`).
5. **Transaction Ledger Integrity**: Forbid recording refunds directly into the `revenue` field as negative numbers. Require separate `refund_amount` and `return_timestamp` columns.